# 2D surface regression with physical SYNE-KANs

This notebook reproduces the 2D function-regression comparison used in the paper. I have deliberately kept the release version narrow: three nonlinear surfaces, one physical KAN architecture, one fixed training recipe, and the corresponding ReLU MLP baselines.

The development Optuna sweeps have already been done. They are not repeated here. This is the code for rerunning the final comparison cleanly, rather than another large hyperparameter-search notebook.

The physical model uses the frozen B1 digital twin supplied with the release. Put `B1_MLP_3_50_50_1.pt` beside this notebook, or point the `SYNE_TWIN_PATH` environment variable to it.


## What this notebook runs

The released task set contains:

1. **Cosine-squares composite**

   $$z=\cos\left(\pi\left[\sin^2(\pi x)+\cos^2(\pi y)\right]\right).$$

2. **Sinc composite**

   $$z=\operatorname{sinc}\left(10\left[\cos^2(\pi x)+|y|\cos(2\pi y)\right]\right),$$

   where $\operatorname{sinc}(u)=\sin(u)/u$.

3. **Exponential-cosine composite**

   $$z=\exp(-|v|)\cos(5v^2),\qquad
   v=\sin^2(\pi x)+\operatorname{sinc}(10y).$$

The physical KAN architecture is always `[2,1,1]`: two inputs, one intermediate neuron, and one output. Twelve SYNE devices per edge is the main configuration. The same optimised recipe is then transferred unchanged to 8, 6, and 4 devices per edge.

The baselines are ordinary ReLU MLPs with one, two, or three hidden layers. We use widths 5, 10, 20, 50, 100, and 200 to cover the relevant parameter range without starting from unnecessarily large networks.

`RUN_PROFILE="quick"` is a short installation check. `RUN_PROFILE="paper"` runs the full device-count and MLP sweeps with three repeats. The two profiles write to separate result folders, so a quick test can never be mistaken for a paper run.


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def stable_seed(*parts) -> int:
    """Build a repeatable integer seed from a readable run description."""
    raw = "|".join(map(str, parts)).encode("utf-8")
    return int(hashlib.sha256(raw).hexdigest()[:8], 16)


def geometric_mean(values) -> float:
    values = np.asarray(values, dtype=float)
    return float(np.exp(np.mean(np.log(np.clip(values, 1e-12, None)))))


print("Device:", DEVICE)
print("PyTorch:", torch.__version__)


In [ ]:
# -----------------------------------------------------------------------------
# USER SETTINGS
# -----------------------------------------------------------------------------

RUN_PROFILE = "quick"       # "quick" or "paper"
RUN_KAN = True
RUN_MLP_BASELINES = True

# The simplest arrangement is to place the twin checkpoint beside this notebook.
# An environment variable is useful on a cluster or when several notebooks share
# one checkpoint.
TWIN_PATH = Path(
    os.environ.get("SYNE_TWIN_PATH", Path.cwd() / "B1_MLP_3_50_50_1.pt")
).expanduser()

OUTPUT_ROOT = Path(
    os.environ.get("SYNE_RESULTS_DIR", f"results_2d_surface_release_{RUN_PROFILE}")
).expanduser()

PROFILES = {
    "quick": {
        "max_epochs": 120,
        "evaluation_seeds": [404],
        "kan_device_counts": [12],
        "mlp_shapes": [(1, 10), (2, 10), (3, 10)],
    },
    "paper": {
        "max_epochs": 800,
        "evaluation_seeds": [404, 505, 606],
        "kan_device_counts": [12, 8, 6, 4],
        "mlp_shapes": [
            (depth, width)
            for depth in (1, 2, 3)
            for width in (5, 10, 20, 50, 100, 200)
        ],
    },
}

if RUN_PROFILE not in PROFILES:
    raise ValueError(f"RUN_PROFILE must be one of {list(PROFILES)}")

PROFILE = PROFILES[RUN_PROFILE]
MAX_EPOCHS = PROFILE["max_epochs"]
EVALUATION_SEEDS = PROFILE["evaluation_seeds"]
KAN_DEVICE_COUNTS = PROFILE["kan_device_counts"]
MLP_SHAPES = PROFILE["mlp_shapes"]

TASKS = [
    "cosine_squares",
    "sinc_composite",
    "exp_cosine_composite",
]

ARCHITECTURE = [2, 1, 1]
N_INPUTS = 2

print("Run profile:", RUN_PROFILE)
print("Twin checkpoint:", TWIN_PATH)
print("Results folder:", OUTPUT_ROOT)
print("Functions:", TASKS)
print("Evaluation seeds:", EVALUATION_SEEDS)
print("KAN devices per edge:", KAN_DEVICE_COUNTS)
print("MLP shapes:", MLP_SHAPES)


## Exact target functions and data protocol

These functions are written directly from the NumPy definitions used to make the paper surfaces. The small `mathematical_sinc()` helper is worth keeping explicit: NumPy defines `np.sinc(q)` as $\sin(\pi q)/(\pi q)$, whereas the paper uses $\sin(u)/u$.

For each repeat, we form an $80\times80$ grid on $[-1,1]^2$, randomly hold out 10% for validation, and train on the remaining 90%. Final MSE is evaluated on a separate $100\times100$ grid. The validation split changes with the repeat seed; the dense test grid is fixed.

The first plot is an intentional sanity check. If these three surfaces do not look right, stop here rather than training the wrong targets for several hours.


In [ ]:
FUNCTION_LABELS = {
    "cosine_squares": r"$\cos\{\pi[\sin^2(\pi x)+\cos^2(\pi y)]\}$",
    "sinc_composite": r"$\mathrm{sinc}\{10[\cos^2(\pi x)+|y|\cos(2\pi y)]\}$",
    "exp_cosine_composite": (
        r"$e^{-|v|}\cos(5v^2)$,  "
        r"$v=\sin^2(\pi x)+\mathrm{sinc}(10y)$"
    ),
}


def mathematical_sinc(value):
    """Return sin(value)/value, including the limiting value 1 at zero."""
    return np.sinc(value / np.pi)


def surface_function(name, x, y):
    if name == "cosine_squares":
        inner = np.sin(np.pi * x) ** 2 + np.cos(np.pi * y) ** 2
        return np.cos(np.pi * inner)

    if name == "sinc_composite":
        inner = np.cos(np.pi * x) ** 2 + np.abs(y) * np.cos(2.0 * np.pi * y)
        return mathematical_sinc(10.0 * inner)

    if name == "exp_cosine_composite":
        v = np.sin(np.pi * x) ** 2 + mathematical_sinc(10.0 * y)
        return np.exp(-np.abs(v)) * np.cos(5.0 * v ** 2)

    raise KeyError(f"Unknown function: {name}")


def make_split(task_name, seed):
    grid = np.linspace(-1, 1, 80, dtype=np.float32)
    xx, yy = np.meshgrid(grid, grid, indexing="xy")
    x_all = np.column_stack([xx.ravel(), yy.ravel()]).astype(np.float32)
    y_all = surface_function(task_name, x_all[:, 0], x_all[:, 1]).astype(np.float32)

    rng = np.random.default_rng(seed)
    order = rng.permutation(len(x_all))
    n_validation = int(0.1 * len(x_all))
    validation_indices = order[:n_validation]
    training_indices = order[n_validation:]

    test_grid = np.linspace(-1, 1, 100, dtype=np.float32)
    test_xx, test_yy = np.meshgrid(test_grid, test_grid, indexing="xy")
    x_test = np.column_stack([test_xx.ravel(), test_yy.ravel()]).astype(np.float32)
    y_test = surface_function(task_name, x_test[:, 0], x_test[:, 1]).astype(np.float32)

    return {
        "x_train": torch.from_numpy(x_all[training_indices]),
        "y_train": torch.from_numpy(y_all[training_indices, None]),
        "x_val": torch.from_numpy(x_all[validation_indices]),
        "y_val": torch.from_numpy(y_all[validation_indices, None]),
        "x_test": torch.from_numpy(x_test),
        "y_test": torch.from_numpy(y_test[:, None]),
        "test_shape": (100, 100),
    }


# Plot the exact arrays used by the training code.
plot_grid = np.linspace(-1, 1, 300, dtype=np.float32)
plot_x, plot_y = np.meshgrid(plot_grid, plot_grid, indexing="xy")
fig, axes = plt.subplots(1, len(TASKS), figsize=(13.2, 3.8), constrained_layout=True)

for axis, task in zip(axes, TASKS):
    target = surface_function(task, plot_x, plot_y)
    if not np.isfinite(target).all():
        raise ValueError(f"Non-finite target values in {task}")
    image = axis.imshow(
        target, origin="lower", extent=[-1, 1, -1, 1],
        cmap="RdBu_r", vmin=-1, vmax=1,
    )
    axis.set_title(FUNCTION_LABELS[task], fontsize=10)
    axis.set_xlabel("x")
    axis.set_ylabel("y")

fig.colorbar(image, ax=axes, shrink=0.86, label="target value")
fig.suptitle("Released 2D regression targets")
plt.show()


## Frozen digital twin and physical KAN

Each KAN edge is a parallel bank of SYNE responses predicted by the frozen B1 digital twin. The twin is never updated by this notebook.

The signal and control sigmoids below are part of the physical parameterisation. They remain in the forward pass during training, validation, and inference; they are not temporary training tricks. Their job is to map unconstrained trainable variables onto the bounded signal and control ranges of the device model.

There are no auxiliary penalties, projection steps, or extra control-voltage initialisation limits in this release recipe. As in the NASA battery release, the KAN uses three learning-rate groups: output gains/biases, control voltages, and signal ranges.

For exact equivalence with the completed experiments, the implementation retains a bias for each simulated device contribution. In the paper count, linear biases entering the same neuron are consolidated into one neuron bias. This gives the reported count

$$N_{\mathrm{paper}}=5D\sum_l n_l n_{l+1}+\sum_l n_{l+1}.$$

For `[2,1,1]`, this is $15D+2$: 62, 92, 122, and 182 parameters for 4, 6, 8, and 12 devices per edge respectively. Those are the KAN counts used in the scaling plot.


In [ ]:
class RebuiltTwin(nn.Module):
    """Fallback for a checkpoint containing only an ordinary MLP state_dict."""
    def __init__(self, weights, biases):
        super().__init__()
        self.weights = nn.ParameterList([nn.Parameter(w.clone(), requires_grad=False) for w in weights])
        self.biases = nn.ParameterList([nn.Parameter(b.clone(), requires_grad=False) for b in biases])

    def forward(self, x):
        for i, (weight, bias) in enumerate(zip(self.weights, self.biases)):
            x = F.linear(x, weight, bias)
            if i + 1 < len(self.weights):
                x = F.relu(x)
        return x

def _natural_key(name: str):
    import re
    return [int(p) if p.isdigit() else p for p in re.split(r"(\d+)", name)]

def _strip_prefix(text: str, prefix: str) -> str:
    """Python 3.8-compatible replacement for str.removeprefix()."""
    return text[len(prefix):] if text.startswith(prefix) else text

def load_frozen_twin(path: Path) -> nn.Module:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Digital twin not found: {path}\n"
            "Set TWIN_PATH in the configuration cell. The expected checkpoint is either "
            "a saved nn.Module, [module, x_scale, y_scale], or an MLP state_dict."
        )
    try:
        payload = torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        # `weights_only` was added in newer PyTorch releases.
        payload = torch.load(path, map_location=DEVICE)
    if isinstance(payload, nn.Module):
        twin = payload
    elif isinstance(payload, (list, tuple)) and payload and isinstance(payload[0], nn.Module):
        twin = payload[0]
    else:
        state = payload.get("model_state_dict", payload.get("state_dict", payload)) if isinstance(payload, dict) else None
        if not isinstance(state, dict):
            raise TypeError(f"Unsupported twin checkpoint payload: {type(payload)}")
        cleaned_state = {}
        for key, value in state.items():
            key = _strip_prefix(str(key), "module.")
            key = _strip_prefix(key, "model.")
            cleaned_state[key] = value
        state = cleaned_state
        weight_keys = sorted([k for k, v in state.items() if k.endswith("weight") and getattr(v, "ndim", 0) == 2], key=_natural_key)
        if not weight_keys:
            raise ValueError("No linear weights found in the twin state_dict")
        weights, biases = [], []
        for key in weight_keys:
            weights.append(state[key].float())
            bias_key = key[:-6] + "bias"
            biases.append(state.get(bias_key, torch.zeros(state[key].shape[0])).float())
        twin = RebuiltTwin(weights, biases)

    twin = twin.to(DEVICE).float().eval()
    for parameter in twin.parameters():
        parameter.requires_grad_(False)
    with torch.no_grad():
        probe = torch.zeros(8, 3, device=DEVICE)
        out = twin(probe).reshape(-1)
    if out.numel() != 8 or not bool(torch.isfinite(out).all()):
        raise ValueError("The twin failed a finite 8x3 input/output probe")
    print("Loaded and froze twin:", path, "|", type(twin).__name__)
    return twin


class PhysicalKAN(nn.Module):
    """Constant-control physical SYNE-KAN used throughout the release codes."""
    def __init__(self, twin, architecture, devices_per_edge, hp):
        super().__init__()
        self.twin = twin
        self.architecture = list(map(int, architecture))
        self.devices_per_edge = int(devices_per_edge)
        self.hp = dict(hp)
        self.controls_per_device = 2
        self.squash_gain = 6.0
        self.output_gain_scale = float(hp["output_gain_scale"])
        self.twin_microbatch = int(hp.get("twin_microbatch", 262144))
        for p in self.twin.parameters():
            p.requires_grad_(False)

        self.S_min, self.S_max = nn.ParameterList(), nn.ParameterList()
        self.V, self.G_o = nn.ParameterList(), nn.ParameterList()
        self.B_o = nn.ParameterList()
        for n_in, n_out in zip(self.architecture[:-1], self.architecture[1:]):
            shape = (n_in, n_out, self.devices_per_edge)
            self.S_min.append(nn.Parameter(torch.zeros(shape, device=DEVICE)))
            self.S_max.append(nn.Parameter(torch.zeros(shape, device=DEVICE)))
            self.V.append(nn.Parameter(torch.zeros(*shape, 2, device=DEVICE)))
            self.G_o.append(nn.Parameter(torch.zeros(shape, device=DEVICE)))
            self.B_o.append(nn.Parameter(torch.zeros(shape, device=DEVICE)))
        self._initialise()

    def train(self, mode=True):
        super().train(mode)
        self.twin.eval()
        return self

    def squash_signal(self, raw):
        return 2.0 * torch.sigmoid(self.squash_gain * raw) - 1.0

    @staticmethod
    def squash_control(raw):
        return 2.0 * torch.sigmoid(raw) - 1.0

    def inverse_signal(self, value):
        p = ((value + 1.0) * 0.5).clamp(1e-6, 1.0 - 1e-6)
        return torch.log(p / (1.0 - p)) / self.squash_gain

    def _initialise(self):
        with torch.no_grad():
            for layer, (n_in, n_out) in enumerate(zip(self.architecture[:-1], self.architecture[1:])):
                shape = (n_in, n_out, self.devices_per_edge)
                count = int(np.prod(shape))
                dv_min = float(self.hp["init_dv_min"])
                dv_max = min(float(self.hp["init_dv_max"]), 1.96)
                widths = torch.linspace(dv_min, dv_max, count, device=DEVICE)
                step = (dv_max - dv_min) / max(count - 1, 1)
                widths += (torch.rand(count, device=DEVICE) - 0.5) * 2 * step * float(self.hp["span_jitter"])
                widths = widths.clamp(dv_min, dv_max)[torch.randperm(count, device=DEVICE)].reshape(shape)
                fan = float(n_in * self.devices_per_edge)
                if self.hp["init_style"] == "powerlaw":
                    widths = dv_min + (widths - dv_min) * max(0.1, min(1.0, fan ** (-float(self.hp["pl_beta"]))))
                inverted = torch.rand(shape, device=DEVICE) < float(self.hp["invert_probability"])
                low, high = -0.98 + 0.5 * widths, 0.98 - 0.5 * widths
                centre = low + torch.rand(shape, device=DEVICE) * (high - low)
                centre = torch.where(inverted, torch.zeros_like(centre), centre)
                vmin, vmax = centre - 0.5 * widths, centre + 0.5 * widths
                lo = torch.where(inverted, vmax, vmin)
                hi = torch.where(inverted, vmin, vmax)
                self.S_min[layer].copy_(self.inverse_signal(lo.clamp(-0.98, 0.98)))
                self.S_max[layer].copy_(self.inverse_signal(hi.clamp(-0.98, 0.98)))

                shrink = fan ** (-float(self.hp["pl_alpha"])) if self.hp["init_style"] == "powerlaw" else 1.0
                limit = math.sqrt(6.0 / 4.0) * float(self.hp["control_init_scale"]) * shrink
                # No lower/upper control-voltage initialisation limit. The raw
                # parameters are sampled symmetrically, then mapped to the
                # normalised physical voltage by squash_control() in forward().
                self.V[layer].uniform_(-limit, limit)
                self.G_o[layer].zero_()
                self.B_o[layer].zero_()

    def _twin_forward(self, flat):
        return torch.cat([self.twin(flat[i:i+self.twin_microbatch]).reshape(-1)
                          for i in range(0, len(flat), self.twin_microbatch)], dim=0)

    def forward(self, inputs):
        activation = inputs
        for layer, (n_in, n_out) in enumerate(zip(self.architecture[:-1], self.architecture[1:])):
            scalar = activation.unsqueeze(2).unsqueeze(3)
            pmin = self.squash_signal(self.S_min[layer])
            pmax = self.squash_signal(self.S_max[layer])
            signal = (0.5 * (pmax-pmin).unsqueeze(0) * scalar + 0.5 * (pmax+pmin).unsqueeze(0)).unsqueeze(-1)
            controls = self.squash_control(self.V[layer]).unsqueeze(0).expand(len(activation), -1, -1, -1, -1)
            twin_in = torch.cat([signal, controls], dim=-1)
            twin_out = self._twin_forward(twin_in.reshape(-1, 3)).reshape(
                len(activation), n_in, n_out, self.devices_per_edge)
            contribution = self.output_gain_scale * self.G_o[layer].unsqueeze(0) * twin_out
            contribution = contribution + self.B_o[layer].unsqueeze(0)
            activation = contribution.sum(dim=(1, 3))
        return activation

    def parameter_groups(self):
        ranges = list(self.S_min.parameters()) + list(self.S_max.parameters())
        controls = list(self.V.parameters())
        outputs = list(self.G_o.parameters()) + list(self.B_o.parameters())
        return [
            {"name": "gains_biases", "params": outputs, "lr": float(self.hp["lr_gobo"]), "weight_decay": 0.0},
            {"name": "controls", "params": controls, "lr": float(self.hp["lr_v"]), "weight_decay": 0.0},
            {"name": "ranges", "params": ranges, "lr": float(self.hp["lr_s"]), "weight_decay": 0.0},
        ]

    def actual_trainable_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def manuscript_parameter_count(self):
        edges = sum(a*b for a, b in zip(self.architecture[:-1], self.architecture[1:]))
        post_synaptic = sum(self.architecture[1:])
        return 5 * edges * self.devices_per_edge + post_synaptic


## MLP baseline and shared training loop

The MLP is a standard fully connected ReLU network with Xavier-initialised weights and zero biases. Both model families use the same train/validation/test splits, the same maximum epoch count, and the same best-validation checkpoint rule.

The KAN uses Adam with the three physical parameter groups defined above. The MLP uses the fixed AdamW recipe selected during development. Early stopping is checked every ten epochs after a minimum of 100 epochs.


In [ ]:
class ReLUMLP(nn.Module):
    def __init__(self, n_in, n_out, width, depth):
        super().__init__()
        dims = [n_in] + [int(width)] * int(depth) + [n_out]
        layers = []
        for i, (a, b) in enumerate(zip(dims[:-1], dims[1:])):
            linear = nn.Linear(a, b)
            nn.init.xavier_uniform_(linear.weight)
            nn.init.zeros_(linear.bias)
            layers.append(linear)
            if i + 1 < len(dims) - 1:
                layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

    def actual_trainable_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

def compact_state(model):
    trainable = {name for name, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in trainable}

def restore_compact_state(model, state):
    current = model.state_dict()
    current.update({k: v.to(current[k].device) for k, v in state.items()})
    model.load_state_dict(current, strict=True)

def make_optimizer(model, hp, is_kan):
    if is_kan:
        return torch.optim.Adam(
            model.parameter_groups(),
            betas=(float(hp["beta1"]), float(hp["beta2"])),
            eps=float(hp.get("adam_eps", 1e-8)),
        )
    cls = torch.optim.AdamW if hp["optimizer"] == "adamw" else torch.optim.Adam
    return cls(model.parameters(), lr=float(hp["lr"]),
               betas=(float(hp["beta1"]), float(hp["beta2"])),
               eps=1e-8, weight_decay=float(hp["weight_decay"]))

def make_scheduler(optimizer, hp, max_epochs):
    if hp["scheduler"] == "cosine":
        minimum = float(hp["lr_min_frac"])
        def cosine_multiplier(epoch):
            progress = min(max(epoch, 0), max_epochs - 1) / float(max(max_epochs - 1, 1))
            return minimum + 0.5 * (1.0 - minimum) * (1.0 + math.cos(math.pi * progress))
        # LambdaLR multiplies every group's own base LR by the same factor,
        # preserving the NASA lr_gobo : lr_v : lr_s ratios throughout training.
        return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=cosine_multiplier)
    if hp["scheduler"] == "step":
        return torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(20, max_epochs // 8), gamma=float(hp["step_gamma"]))
    return None

@torch.no_grad()
def predict(model, x, chunk=8192):
    model.eval()
    outs = []
    for i in range(0, len(x), chunk):
        outs.append(model(x[i:i+chunk].to(DEVICE)).detach().cpu())
    return torch.cat(outs)

def train_one(model, split, hp, problem, max_epochs, seed, verbose=False):
    seed_everything(seed)
    model = model.to(DEVICE)
    optimizer = make_optimizer(model, hp, isinstance(model, PhysicalKAN))
    scheduler = make_scheduler(optimizer, hp, max_epochs)
    loss_fn = nn.BCEWithLogitsLoss() if problem == "classification" else nn.MSELoss()
    loader = DataLoader(TensorDataset(split["x_train"], split["y_train"]),
                        batch_size=int(hp["batch_size"]), shuffle=True,
                        generator=torch.Generator().manual_seed(seed), pin_memory=torch.cuda.is_available())
    best_loss, best_epoch, best_state = float("inf"), 0, None
    evaluations_without_improvement = 0
    history = []
    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = loss_fn(pred.reshape_as(yb), yb)
            loss.backward()
            if float(hp["grad_clip"]) > 0:
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], float(hp["grad_clip"]))
            optimizer.step()
        if scheduler is not None:
            scheduler.step()
        if epoch == 1 or epoch % int(hp["evaluate_every"]) == 0 or epoch == max_epochs:
            val_pred = predict(model, split["x_val"])
            val_loss = float(loss_fn(val_pred.reshape_as(split["y_val"]), split["y_val"]).item())
            history.append((epoch, val_loss))
            if val_loss < best_loss - float(hp["min_delta"]):
                best_loss, best_epoch, best_state = val_loss, epoch, compact_state(model)
                evaluations_without_improvement = 0
            else:
                evaluations_without_improvement += 1
            if verbose:
                print(f"epoch={epoch:4d} val_loss={val_loss:.6g} best={best_loss:.6g}@{best_epoch}")
            if epoch >= int(hp["minimum_epochs"]) and evaluations_without_improvement >= int(hp["patience_evaluations"]):
                break
    restore_compact_state(model, best_state)
    val_pred = predict(model, split["x_val"])
    if problem == "classification":
        val_metric = float(((val_pred.reshape(-1) >= 0) == (split["y_val"].reshape(-1) >= 0.5)).float().mean())
    else:
        val_metric = float(F.mse_loss(val_pred.reshape_as(split["y_val"]), split["y_val"]))
    return model, {"val_loss": best_loss, "val_metric": val_metric, "best_epoch": best_epoch, "history": history}


## Fixed release recipes

The dictionaries below are the final recipes from the completed development notebook. Keeping them explicit makes the release run auditable and avoids quietly tuning on the evaluation repeats.

The D12 KAN recipe is the main recipe. D8, D6, and D4 change only the number of devices per edge. Likewise, every MLP width/depth pair uses the same optimiser recipe.


In [ ]:
KAN_RECIPE = {
    "batch_size": 64,
    "beta1": 0.7528349884567577,
    "beta2": 0.9966047298423021,
    "control_init_scale": 1.4102077428899398,
    "evaluate_every": 10,
    "grad_clip": 3.964976397415234,
    "init_dv_min": 1.215482562462976,
    "init_dv_max": 1.3912415175217472,
    "init_style": "xavier",
    "invert_probability": 0.41231365992504,
    "lr_gobo": 0.003450754115363201,
    "lr_min_frac": 0.019288775523797416,
    "lr_s": 0.0012477461433447496,
    "lr_v": 0.0010597080877138103,
    "min_delta": 1e-8,
    "minimum_epochs": 100,
    "output_gain_scale": 50.0,
    "patience_evaluations": 20,
    "pl_alpha": 0.08552353444369976,
    "pl_beta": 0.05792571407734193,
    "scheduler": "none",
    "span_jitter": 0.132244255395231,
    "step_gamma": 0.8213486121877613,
    "adam_eps": 1e-8,
}

MLP_RECIPE = {
    "batch_size": 256,
    "beta1": 0.7557942523857278,
    "beta2": 0.8706569402574451,
    "evaluate_every": 10,
    "grad_clip": 2.078326868397085,
    "lr": 0.0019567725431816224,
    "lr_min_frac": 0.03669072341844597,
    "min_delta": 1e-8,
    "minimum_epochs": 100,
    "optimizer": "adamw",
    "patience_evaluations": 20,
    "scheduler": "none",
    "step_gamma": 0.6838862823457738,
    "weight_decay": 1.1445840775827944e-8,
}

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / "predictions").mkdir(parents=True, exist_ok=True)

TWIN = load_frozen_twin(TWIN_PATH)

print("\nReported KAN parameter counts:")
for devices in KAN_DEVICE_COUNTS:
    probe_hp = dict(KAN_RECIPE, devices_per_edge=devices)
    probe = PhysicalKAN(TWIN, ARCHITECTURE, devices, probe_hp)
    print(f"  [2,1,1], D={devices:2d}: {probe.manuscript_parameter_count():4d}")
    del probe

release_manifest = {
    "run_profile": RUN_PROFILE,
    "tasks": TASKS,
    "architecture": ARCHITECTURE,
    "kan_device_counts": KAN_DEVICE_COUNTS,
    "evaluation_seeds": EVALUATION_SEEDS,
    "max_epochs": MAX_EPOCHS,
    "kan_recipe": KAN_RECIPE,
    "mlp_recipe": MLP_RECIPE,
    "torch_version": torch.__version__,
    "numpy_version": np.__version__,
}
(OUTPUT_ROOT / "release_manifest.json").write_text(
    json.dumps(release_manifest, indent=2), encoding="utf-8"
)


## Resumable result logging

Every completed run is appended immediately to `runs.csv`. If a long paper sweep is interrupted, rerun the notebook with the same profile and it will skip completed run IDs.

Predictions are saved for the D12 KAN and one representative MLP so that the reconstruction plots do not require retraining.


In [ ]:
RESULTS_PATH = OUTPUT_ROOT / "runs.csv"


def prediction_path(model_family, task, configuration, seed):
    filename = f"{model_family}_{task}_{configuration}_seed{seed}.npz".replace("/", "_")
    return OUTPUT_ROOT / "predictions" / filename


if RESULTS_PATH.exists():
    RUN_ROWS = pd.read_csv(RESULTS_PATH).to_dict("records")
else:
    RUN_ROWS = []

COMPLETED_RUN_IDS = {str(row["run_id"]) for row in RUN_ROWS}


def save_run_rows():
    pd.DataFrame(RUN_ROWS).to_csv(RESULTS_PATH, index=False)


def run_and_record(
    model_family,
    task,
    seed,
    configuration,
    architecture,
    hp,
    build_model,
    reported_parameters,
    devices=None,
    depth=None,
    width=None,
    save_prediction=False,
):
    run_id = "|".join(map(str, [model_family, task, seed, configuration]))
    if run_id in COMPLETED_RUN_IDS:
        return False

    run_seed = stable_seed("2d_surface_release", run_id)
    seed_everything(run_seed)  # Seed before constructing the model.
    model = build_model()
    split = make_split(task, seed)

    started = time.time()
    model, result = train_one(model, split, hp, "regression", MAX_EPOCHS, run_seed)
    elapsed_seconds = time.time() - started

    test_prediction = predict(model, split["x_test"])
    test_mse = float(
        F.mse_loss(test_prediction.reshape_as(split["y_test"]), split["y_test"])
    )

    row = {
        "run_id": run_id,
        "model_family": model_family,
        "task": task,
        "seed": seed,
        "configuration": configuration,
        "architecture": architecture,
        "devices_per_edge": devices,
        "depth": depth,
        "width": width,
        "parameters": int(reported_parameters),
        "validation_mse": float(result["val_metric"]),
        "test_mse": test_mse,
        "best_epoch": int(result["best_epoch"]),
        "elapsed_seconds": elapsed_seconds,
    }

    if save_prediction:
        np.savez_compressed(
            prediction_path(model_family, task, configuration, seed),
            x=split["x_test"].numpy(),
            target=split["y_test"].numpy().reshape(-1),
            prediction=test_prediction.numpy().reshape(-1),
        )

    RUN_ROWS.append(row)
    COMPLETED_RUN_IDS.add(run_id)
    save_run_rows()
    print(row)

    del model
    return True


## Run the final KAN and MLP sweeps

The ordering here is intentional: D12 is run first because it is the main physical KAN configuration. The smaller device banks follow as recipe-transfer tests.

Nothing is selected using these test results. Each model is trained on the training grid, checkpointed using validation MSE, and evaluated once on the independent dense test grid.


In [ ]:
# Choose the largest active three-layer MLP for the reconstruction comparison.
VISUAL_MLP_SHAPE = max(
    MLP_SHAPES,
    key=lambda shape: (shape[0], shape[1]),
)


if RUN_KAN:
    for devices in KAN_DEVICE_COUNTS:
        hp = dict(KAN_RECIPE, devices_per_edge=devices)
        probe = PhysicalKAN(TWIN, ARCHITECTURE, devices, hp)
        reported_parameters = probe.manuscript_parameter_count()
        del probe

        configuration = f"KAN_2_1_1_D{devices}"
        for task in TASKS:
            for seed in EVALUATION_SEEDS:
                run_and_record(
                    model_family="KAN",
                    task=task,
                    seed=seed,
                    configuration=configuration,
                    architecture="[2,1,1]",
                    hp=hp,
                    build_model=lambda d=devices, h=hp: PhysicalKAN(
                        TWIN, ARCHITECTURE, d, h
                    ),
                    reported_parameters=reported_parameters,
                    devices=devices,
                    save_prediction=(devices == 12 and seed == EVALUATION_SEEDS[0]),
                )


if RUN_MLP_BASELINES:
    for depth, width in MLP_SHAPES:
        hp = dict(MLP_RECIPE, depth=depth, width=width)
        probe = ReLUMLP(N_INPUTS, 1, width, depth)
        parameter_count = probe.actual_trainable_parameters()
        del probe

        shape_tag = f"L{depth}_H{width}"
        configuration = f"MLP_{shape_tag}"
        architecture = "[" + ",".join(map(str, [2] + [width] * depth + [1])) + "]"

        for task in TASKS:
            for seed in EVALUATION_SEEDS:
                run_and_record(
                    model_family="MLP",
                    task=task,
                    seed=seed,
                    configuration=configuration,
                    architecture=architecture,
                    hp=hp,
                    build_model=lambda d=depth, w=width: ReLUMLP(N_INPUTS, 1, w, d),
                    reported_parameters=parameter_count,
                    depth=depth,
                    width=width,
                    save_prediction=(
                        (depth, width) == VISUAL_MLP_SHAPE
                        and seed == EVALUATION_SEEDS[0]
                    ),
                )


expected_runs = (
    (len(KAN_DEVICE_COUNTS) if RUN_KAN else 0)
    + (len(MLP_SHAPES) if RUN_MLP_BASELINES else 0)
) * len(TASKS) * len(EVALUATION_SEEDS)

print(f"Completed rows in this result folder: {len(RUN_ROWS)} / {expected_runs}")


## Summaries and parameter–performance scaling

For each configuration, we first take the geometric mean MSE across the three functions within each repeat. We then take the geometric mean over repeat seeds. The plotted range is the minimum to maximum repeat-level geometric mean.

Incomplete configurations are excluded rather than being allowed to look artificially good because an awkward function or repeat has not finished. `rollout_completion_audit.csv` records the coverage explicitly.


In [ ]:
run_table = pd.read_csv(RESULTS_PATH)
expected_per_configuration = len(TASKS) * len(EVALUATION_SEEDS)

coverage = (
    run_table.groupby(["model_family", "configuration"])
    .agg(
        rows=("run_id", "size"),
        functions=("task", "nunique"),
        seeds=("seed", "nunique"),
    )
    .reset_index()
)

complete_configurations = coverage[
    (coverage["rows"] == expected_per_configuration)
    & (coverage["functions"] == len(TASKS))
    & (coverage["seeds"] == len(EVALUATION_SEEDS))
][["model_family", "configuration"]]

complete_rows = run_table.merge(
    complete_configurations,
    on=["model_family", "configuration"],
    how="inner",
)

function_summary = (
    complete_rows.groupby(
        ["model_family", "configuration", "task"], dropna=False
    )
    .agg(
        geometric_test_mse=("test_mse", geometric_mean),
        seed_min=("test_mse", "min"),
        seed_max=("test_mse", "max"),
        parameters=("parameters", "first"),
    )
    .reset_index()
)

per_seed = (
    complete_rows.groupby(
        ["model_family", "configuration", "seed"], dropna=False
    )
    .agg(
        seed_geometric_mse=("test_mse", geometric_mean),
        architecture=("architecture", "first"),
        devices=("devices_per_edge", "first"),
        depth=("depth", "first"),
        width=("width", "first"),
        parameters=("parameters", "first"),
    )
    .reset_index()
)

configuration_summary = (
    per_seed.groupby(["model_family", "configuration"], dropna=False)
    .agg(
        geometric_test_mse=("seed_geometric_mse", geometric_mean),
        seed_min=("seed_geometric_mse", "min"),
        seed_max=("seed_geometric_mse", "max"),
        architecture=("architecture", "first"),
        devices=("devices", "first"),
        depth=("depth", "first"),
        width=("width", "first"),
        parameters=("parameters", "first"),
    )
    .reset_index()
)

coverage.to_csv(OUTPUT_ROOT / "rollout_completion_audit.csv", index=False)
function_summary.to_csv(OUTPUT_ROOT / "summary_by_function.csv", index=False)
configuration_summary.to_csv(
    OUTPUT_ROOT / "summary_by_configuration.csv", index=False
)

print(
    configuration_summary.sort_values(
        ["model_family", "depth", "parameters"], na_position="first"
    ).to_string(index=False)
)


colors = {"KAN": "#0b3c6f", 1: "#f39c12", 2: "#e67e22", 3: "#c65d00"}
markers = {"KAN": "X", 1: "^", 2: "s", 3: "D"}

fig, axis = plt.subplots(figsize=(9.2, 5.8))

kan_part = configuration_summary[
    configuration_summary["model_family"] == "KAN"
].sort_values("parameters")
if len(kan_part):
    y = kan_part["geometric_test_mse"].to_numpy(float)
    axis.errorbar(
        kan_part["parameters"],
        y,
        yerr=np.vstack([y - kan_part["seed_min"], kan_part["seed_max"] - y]),
        marker=markers["KAN"],
        color=colors["KAN"],
        linewidth=2.2,
        capsize=3,
        label="Physical KAN [2,1,1]",
    )

for depth in (1, 2, 3):
    mlp_part = configuration_summary[
        (configuration_summary["model_family"] == "MLP")
        & (configuration_summary["depth"] == depth)
    ].sort_values("parameters")
    if len(mlp_part):
        y = mlp_part["geometric_test_mse"].to_numpy(float)
        axis.errorbar(
            mlp_part["parameters"],
            y,
            yerr=np.vstack([y - mlp_part["seed_min"], mlp_part["seed_max"] - y]),
            marker=markers[depth],
            color=colors[depth],
            linestyle="--",
            linewidth=1.8,
            capsize=3,
            label=f"MLP, {depth} hidden layer{'s' if depth > 1 else ''}",
        )

axis.set_xscale("log")
axis.set_yscale("log")
axis.set_xlabel("Network size / trainable parameters (paper count for KAN)")
axis.set_ylabel("Geometric mean test MSE over three functions")
axis.set_title("2D function regression: parameter–performance scaling")
axis.grid(True, which="both", alpha=0.25)
axis.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "parameter_performance_scaling.png", dpi=240)
plt.show()


## Surface reconstructions

Finally, we plot the ground truth beside the D12 physical KAN and the largest active three-layer MLP. These plots use saved predictions from the first evaluation repeat. The test MSE is written into each prediction title so that the image and the numerical result stay together.


In [ ]:
visual_seed = EVALUATION_SEEDS[0]
visual_depth, visual_width = VISUAL_MLP_SHAPE
visual_mlp_configuration = f"MLP_L{visual_depth}_H{visual_width}"

for task in TASKS:
    kan_path = prediction_path("KAN", task, "KAN_2_1_1_D12", visual_seed)
    mlp_path = prediction_path("MLP", task, visual_mlp_configuration, visual_seed)

    if not (kan_path.exists() and mlp_path.exists()):
        print("Skipping incomplete reconstruction:", task)
        continue

    kan_data = np.load(kan_path)
    mlp_data = np.load(mlp_path)
    shape = (100, 100)

    target = kan_data["target"].reshape(shape)
    kan_prediction = kan_data["prediction"].reshape(shape)
    mlp_prediction = mlp_data["prediction"].reshape(shape)

    kan_mse = float(np.mean((kan_prediction - target) ** 2))
    mlp_mse = float(np.mean((mlp_prediction - target) ** 2))

    fields = [target, kan_prediction, mlp_prediction]
    titles = [
        "Ground truth",
        f"Physical KAN [2,1,1], D12\nMSE = {kan_mse:.3e}",
        f"MLP L{visual_depth}, H{visual_width}\nMSE = {mlp_mse:.3e}",
    ]

    lower, upper = float(target.min()), float(target.max())
    fig, axes = plt.subplots(1, 3, figsize=(11.4, 3.5), constrained_layout=True)
    for axis, field, title in zip(axes, fields, titles):
        image = axis.imshow(
            field,
            origin="lower",
            extent=[-1, 1, -1, 1],
            cmap="RdBu_r",
            vmin=lower,
            vmax=upper,
        )
        axis.set_title(title)
        axis.set_xlabel("x")
        axis.set_ylabel("y")

    fig.colorbar(image, ax=axes, shrink=0.82)
    fig.suptitle(FUNCTION_LABELS[task])
    fig.savefig(OUTPUT_ROOT / f"{task}_reconstruction.png", dpi=220)
    plt.show()


## Files written by a complete run

- `runs.csv`: one row per model, function, and repeat
- `summary_by_function.csv`: repeat-aggregated MSE for each function
- `summary_by_configuration.csv`: geometric-mean MSE over all three functions
- `rollout_completion_audit.csv`: explicit completeness check
- `release_manifest.json`: profile, seeds, software versions, and fixed recipes
- `predictions/*.npz`: saved dense-grid predictions used by the reconstruction plots
- `parameter_performance_scaling.png` and one reconstruction PNG per function

The result folder is deliberately plain. It should be possible to inspect the complete numerical record without needing to reopen the notebook.
